# Model Evaluation Report
This notebook isolates the core evaluation metrics and visualization logic. Data is loaded directly from our upstream ML pipeline's serialized output.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
from IPython.display import display
import pickle

## Pipeline Artifact Injection

In [2]:
# Load the evaluated pipeline results
with open('model_evaluation_results.pkl', 'rb') as f:
    eval_data = pickle.load(f)

model_name = eval_data['model_name']
problem_type = eval_data['problem_type']
X = eval_data['X']
results = eval_data['results']
target_col = eval_data['target_col']

## 1. Evaluation Metrics Calculation

In [3]:
if problem_type == 'regression':
    print(f"RMSE: {results.get('rmse', 0):.4f}")
    print(f"R² Score: {results.get('r2', 0):.4f}")
elif problem_type == 'classification':
    print(f"Accuracy: {results.get('accuracy', 0):.4f}")
    cm = results.get('cm')
    if cm:
        print("Confusion Matrix:")
        display(pd.DataFrame(cm))
elif problem_type == 'clustering':
    print(f"Silhouette Score: {results.get('silhouette', 0):.4f}")

RMSE: 47463.6436
R² Score: -0.0854


## 2. Plots & Visualizations

In [4]:
if problem_type == 'regression':
    y_test = results.get('y_test')
    y_pred = results.get('y_pred')
    if y_test is not None and y_pred is not None:
        fig = px.scatter(x=y_test, y=y_pred, labels={'x': 'Actual', 'y': 'Predicted'}, title="Actual vs Predicted")
        fig.add_shape(type="line", x0=min(y_test), x1=max(y_test), y0=min(y_test), y1=max(y_test), line=dict(dash="dash", color="red"))
        fig.show()
        
elif problem_type == 'classification':
    cm = results.get('cm')
    if cm:
        fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title="Confusion Matrix Heatmap")
        fig.show()
        
elif problem_type == 'clustering':
    preds = results.get('y_pred')
    if preds is not None:
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)
        fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], color=[str(c) for c in preds], title="Cluster Scatter Plot (PCA 2D)")
        fig.show()
        
    if model_name == 'hierarchical':
        fig, ax = plt.subplots(figsize=(10, 5))
        Z = linkage(X.head(100), 'ward')
        dendrogram(Z, ax=ax)
        plt.show()

## 3. Feature Importance Analysis

In [5]:
if 'feature_importance' in results and len(results['feature_importance']) > 0:
    imp_df = pd.DataFrame(list(results['feature_importance'].items())[:10], columns=['Feature', 'Value'])
    fig = px.bar(imp_df, x='Value', y='Feature', orientation='h', title="Top 10 Feature Importance")
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()